# ML-10 — Action Playbook
**Lane 2 — Refresh / Content Opportunity Scoring**

Builds the ranked, reason-coded refresh queue from the winning model
(Logistic Regression, see `w05_model.ipynb`), tiers it into
`refresh_priority` / `refresh_backlog` / `no_action`, and checks it for the same
per-client concentration weakness the baseline had.

In [1]:
import pandas as pd, numpy as np, pickle, json
from pathlib import Path

test = pd.read_pickle("../artifacts/test_set.pkl")
fitted = pickle.load(open("../artifacts/fitted_models.pkl", "rb"))
print(f"Test set: {len(test):,} pages, {test['client_id'].nunique()} clients")
print("Columns with model scores already attached:",
      [c for c in test.columns if c.startswith('proba_')])

Test set: 7,115 pages, 8 clients
Columns with model scores already attached: ['proba_logistic_regression', 'proba_random_forest']


## Tier the queue: top decile = priority, next 30% = backlog, rest = no action

In [2]:
score_col = "proba_logistic_regression"
q90 = test[score_col].quantile(0.90)
q60 = test[score_col].quantile(0.60)
test["action"] = np.select(
    [test[score_col] >= q90, test[score_col] >= q60],
    ["refresh_priority", "refresh_backlog"], default="no_action")

print(test["action"].value_counts())

action
no_action           4269
refresh_backlog     2134
refresh_priority     712
Name: count, dtype: int64


## Reason codes (why a page was flagged, in plain terms an editor can act on)

In [3]:
def reason_code(row):
    r = []
    if row["freshness_tier"] in ("91-180", "181+"):
        r.append("stale")
    if row["position_tier"] in ("page_1", "striking"):
        r.append("strikable_position")
    if row["impression_tier"] in ("moderate", "good", "excellent"):
        r.append("visible")
    if row["days_with_impressions"] < 45:
        r.append("inconsistent_visibility")
    return "+".join(r) if r else "low_signal"

test["reason_code"] = test.apply(reason_code, axis=1)
print(test["reason_code"].value_counts().head(8))

reason_code
strikable_position+visible                          2088
strikable_position+inconsistent_visibility          1694
visible                                              682
inconsistent_visibility                              655
strikable_position                                   536
stale+strikable_position+visible                     381
stale+strikable_position+inconsistent_visibility     330
stale+visible                                        305
Name: count, dtype: int64


## Ranked queue (the actual deliverable an editor opens)

In [4]:
cols = ["content_id","client_id",score_col,"action","reason_code",
        "impressions_90d","avg_position","position_tier","freshness_tier",
        "days_since_last_update","content_type","main_intent","is_declining_label"]
queue = (test[cols]
         .rename(columns={score_col: "model_score"})
         .sort_values("model_score", ascending=False)
         .reset_index(drop=True))
Path("../artifacts").mkdir(exist_ok=True)
queue.to_csv("../artifacts/model_ranked_queue_test.csv", index=False)
queue.head(10)[["content_id","client_id","model_score","action","reason_code","is_declining_label"]]

,content_id,client_id,model_score,action,reason_code,is_declining_label
0,content_a928cb66d230,client_f369cb89fc,0.971268,refresh_priority,strikable_position,1
1,content_7be5f150dc65,client_f369cb89fc,0.966379,refresh_priority,strikable_position,0
2,content_a8864e189b2e,client_d029fa3a95,0.965735,refresh_priority,strikable_position,1
3,content_5d77d3077984,client_f369cb89fc,0.959252,refresh_priority,strikable_position+inconsistent_visibility,1
4,content_87c007fb5c26,client_f369cb89fc,0.952377,refresh_priority,strikable_position+visible,1
5,content_b5e9e6453511,client_f369cb89fc,0.951314,refresh_priority,strikable_position,1
6,content_c82bc0c24241,client_f369cb89fc,0.949972,refresh_priority,strikable_position+visible,1
7,content_ff102de380d8,client_f369cb89fc,0.947525,refresh_priority,strikable_position,1
8,content_96dba8ca02c1,client_f369cb89fc,0.946452,refresh_priority,strikable_position,1
9,content_374e795aab68,client_f369cb89fc,0.945569,refresh_priority,low_signal,0


## Known limitation: per-client concentration

Same weakness the baseline (`w04`) had — a raw score sort over-represents a few clients
at the top of the queue. A production version needs a per-client quota, not just a global
sort (see paper §6, recommendation 2).

In [5]:
top50 = queue.head(50)
tp = int(top50["is_declining_label"].sum())
print(f"Top-50 precision: {tp}/50 = {tp/50:.0%}  (matches model_results.json precision_at_50)")
print()
print("Client concentration in top-50:")
print(top50["client_id"].value_counts())

Top-50 precision: 37/50 = 74%  (matches model_results.json precision_at_50)

Client concentration in top-50:
client_id
client_f369cb89fc    22
client_d029fa3a95    13
client_4e07408562     8
client_8527a891e2     7
Name: count, dtype: int64


In [6]:
json.dump({
    "action_counts": test["action"].value_counts().to_dict(),
    "top50_precision": tp/50,
    "top50_client_concentration": top50["client_id"].value_counts().to_dict(),
}, open("../artifacts/playbook_summary.json", "w"), indent=2)
print("saved playbook_summary.json and model_ranked_queue_test.csv")

saved playbook_summary.json and model_ranked_queue_test.csv
